# SNA Group 14 — 00 · Tạo lại Amazon-book Semantic Data bằng Gemini + OpenAI

**Đây là pipeline DỮ LIỆU mới, không viết lại E1–E4.** Dữ liệu gốc + mapping RLMRec → *Gemini qua DFLO* tạo item/user profiles → *OpenAI `text-embedding-ada-002`* mã hóa trường `summarization` → xuất 7 file tương thích RLMRec.

**Thứ tự:** 0–4 (chuẩn bị nguồn và audit) → 5 (item prompts) → 6 (item pilot/full) → 7 (user prompts) → 8 (user pilot/full) → 9 (embeddings) → 10 (package & QA) → mở notebook `01_PREPARE_GROUP_EXPERIMENTS_Colab.ipynb`.

**An toàn:** Không sửa `Data_SNA_Group14/data/raw/amazon`, không sử dụng profiles/embeddings do paper cấp làm đầu vào Gemini, không ghi đè thí nghiệm cũ. Mọi API hàng loạt đều **mặc định khóa**; không chạy `Run all` khi chưa điều chỉnh cấu hình.

**Ranh giới phương pháp:** Giữ nguyên system instructions Amazon trong GitHub RLMRec. Code công khai chỉ có script minh họa tạo profile, không có đầy đủ code khôi phục prompt Amazon hàng loạt. Các quy tắc chọn lịch sử train, xử lý metadata thiếu và review khớp train là phần **reconstruction của nhóm**, có provenance và giới hạn dữ liệu event-level.

**Nguồn code:** [Paper repository](https://github.com/HKUDS/RLMRec) commit `22413752246de3dee8ab0d509f7f7a8889080f95` · [Group repository](https://github.com/trungkien1502/project_sna_group14) commit `c17caeb4448a8add7939510411339af918d258bb`.


### 0. Cài thư viện và Mount Drive — chạy một lần mỗi phiên Colab

In [ ]:

%pip -q install openai pyyaml scipy numpy pandas
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, hashlib, gzip, ast, html, re, pickle, time, subprocess, shutil
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import scipy.sparse as sp
from getpass import getpass
from openai import OpenAI
print('READY: Python', sys.version.split()[0])


### 1. Cấu hình đường dẫn, model, giới hạn tiền — chỉ chỉnh ô này

**Raw reviews phải là dữ liệu review thực sự từ Amazon Books 2014** (có `reviewerID`, `asin`, `reviewText`); không lấy reviews từ `.pkl` semantic tác giả. Mặc định từ chối tạo bộ dữ liệu chính nếu chưa có reviews. Nếu chỉ muốn pilot kiểm tra kỹ thuật, đặt `REQUIRE_REVIEWS=False` nhưng **không được gọi kết quả đó là bám đầy đủ quy trình paper**.

In [ ]:

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
RUN_ID = 'gemini_dflo_ada002_v1'  # đổi nếu bắt đầu một protocol khác
WORK = ROOT / 'experiments_gemini' / RUN_ID
RAW = ROOT / 'data/raw/amazon'
EXTERNAL = ROOT / 'data/external/amazon_books'
RAW_METADATA = EXTERNAL / 'meta_Books.json.gz'
RAW_REVIEWS = EXTERNAL / 'reviews_Books.json.gz'  # bản đầy đủ đã audit: 120464/120464 train pairs
REQUIRE_REVIEWS = True             # không âm thầm bỏ review trong thí nghiệm chính
MAX_DESCRIPTION_CHARS = 3200
MAX_REVIEW_CHARS = 650
MAX_USER_HISTORY = None             # paper-style: ALL train items; set integer cap only if needed (new RUN_ID)
MISSING_DESC_REVIEW_COUNT = 3      # reconstruction: reviews từ train pairs nếu thiếu metadata description

DFLO_BASE_URL = 'https://llm.dflo.ai/v1'
DFLO_MODEL = 'ag/gemini-3.8-flash' # alias bro đã test; lưu response.model thực trả về
OPENAI_EMBED_MODEL = 'text-embedding-ada-002'

# Giá dflo chưa được xác minh. Điền số đã kiểm tra tại dflo nếu muốn dự toán.
DFLO_INPUT_USD_PER_M = None
DFLO_OUTPUT_USD_PER_M = None
ADA002_USD_PER_M = 0.10  # OpenAI API trực tiếp; nên xác minh lại trước khi chạy

# Không cho phép API từ notebook khi chưa đổi 2 cờ này.
ALLOW_DFLO_CALLS = False
ALLOW_OPENAI_CALLS = False
MIN_SECONDS_BETWEEN_DFLO_CALLS = 1.0 # thay theo quota dflo

WORK.mkdir(parents=True,exist_ok=True)
for d in ('audit','raw_index','prompts','profiles','embedding_parts','dataset/amazon','logs'):
    (WORK/d).mkdir(parents=True,exist_ok=True)
assert ROOT.is_dir() and RAW.is_dir(), 'Không thấy Drive hoặc dữ liệu gốc'
print('RUN ROOT:',WORK)
print('Raw metadata exists:',RAW_METADATA.is_file())
print('Raw reviews exists:',RAW_REVIEWS.is_file())


### 2. Clone mã nguồn đúng commit & audit train/val/test/mapping

In [ ]:

PAPER_REPO = Path('/content/RLMRec_GeminiData_pinned')
PAPER_COMMIT = '22413752246de3dee8ab0d509f7f7a8889080f95'
if not PAPER_REPO.exists():
    subprocess.run(['git','clone','https://github.com/HKUDS/RLMRec.git',str(PAPER_REPO)],check=True)
have = subprocess.check_output(['git','-C',str(PAPER_REPO),'rev-parse','HEAD'],text=True).strip()
if have != PAPER_COMMIT:
    assert not subprocess.check_output(['git','-C',str(PAPER_REPO),'status','--porcelain'],text=True).strip(), 'Paper repo dirty'
    subprocess.run(['git','-C',str(PAPER_REPO),'fetch','origin',PAPER_COMMIT],check=True)
    subprocess.run(['git','-C',str(PAPER_REPO),'checkout','--detach',PAPER_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(PAPER_REPO),'rev-parse','HEAD'],text=True).strip()==PAPER_COMMIT

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return h.hexdigest()
def digest(s):return hashlib.sha256(s.encode('utf-8')).hexdigest()
def read_jsonl(path):
    with Path(path).open('r',encoding='utf-8') as f:
        for i,s in enumerate(f,1):
            if s.strip():
                try: yield json.loads(s)
                except Exception as exc: raise ValueError(f'Invalid JSONL line {i} in {path}') from exc
def atomic_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    tmp.replace(path)
def require_or_freeze(path,payload):
    path=Path(path)
    if path.is_file():
        old=json.loads(path.read_text(encoding='utf-8'))
        assert old == payload, f'Run locked with different inputs: {path}. Đổi RUN_ID.'
    else:atomic_json(path,payload)
def text_rows(p):
    op=gzip.open if str(p).endswith('.gz') else open
    with op(p,'rt',encoding='utf-8',errors='replace') as f:
        for lineno,line in enumerate(f,1):
            if not line.strip():continue
            try:yield json.loads(line)
            except json.JSONDecodeError:
                try: yield ast.literal_eval(line) # meta_Books 2014 thường là Python literals
                except Exception as exc:raise ValueError(f'{p} line {lineno}') from exc
def clean_text(s,lim):
    if s is None:s=''
    if isinstance(s,(list,tuple)):
        def flatten(z):
            if isinstance(z,(list,tuple)):return ' '.join(flatten(v) for v in z)
            return str(z)
        s=flatten(s)
    s=html.unescape(re.sub(r'<[^>]*>',' ',str(s)))
    s=re.sub(r'\s+',' ',s).strip()
    return s[:lim] if s else 'None'

def get_mapper(name,identity_key,index_key,n):
    path=PAPER_REPO/'data/mapper'/name
    dic={};reverse={}
    for row in read_jsonl(path):
        k=int(row[index_key]);v=str(row[identity_key])
        assert k not in dic and v not in reverse,(name,k,v)
        dic[k]=v;reverse[v]=k
    assert set(dic)==set(range(n)),(name,len(dic),n)
    return dic,reverse,path

SPLITS=[]
for name in ['trn_mat.pkl','val_mat.pkl','tst_mat.pkl']:
    with (RAW/name).open('rb') as f: x=pickle.load(f) # chỉ load file .pkl tin cậy của nhóm
    assert sp.issparse(x),name
    x=x.tocsr();x.sum_duplicates();x=(x!=0).astype(np.uint8).tocsr();x.eliminate_zeros()
    SPLITS.append(x)
TRN,VAL,TST=SPLITS
N_USERS,N_ITEMS=TRN.shape
assert (N_USERS,N_ITEMS)==(11000,9332),(N_USERS,N_ITEMS)
assert [x.nnz for x in SPLITS]==[120464,40290,40106]
assert all(x.shape==TRN.shape for x in SPLITS)
assert TRN.multiply(VAL).nnz==TRN.multiply(TST).nnz==VAL.multiply(TST).nnz==0
ITEM_IDS,ASIN_TO_IID,ITEM_MAP_FILE=get_mapper('amazon_item.json','asin','iid',N_ITEMS)
USER_IDS,REVIEWER_TO_UID,USER_MAP_FILE=get_mapper('amazon_user.json','reviewerID','uid',N_USERS)
split_hashes={n:sha256(RAW/n) for n in ['trn_mat.pkl','val_mat.pkl','tst_mat.pkl']}
split_hashes.update({'amazon_item.json':sha256(ITEM_MAP_FILE),'amazon_user.json':sha256(USER_MAP_FILE)})
audit={'status':'PASS','shapes':[N_USERS,N_ITEMS],
       'nnz':[int(x.nnz) for x in SPLITS],'split_and_mapper_sha256':split_hashes,
       'note':'Kiểm tra split, không chứng minh provenance của raw reviews'}
require_or_freeze(WORK/'audit/splits_locked.json',audit)
print('AUDIT PASS',audit['shapes'],audit['nnz'])


### 3. Chuẩn bị raw reviews (chỉ khi chưa có file)

Mình đã có `meta_Books.json.gz` từ trước. **Reviews 2014 có dung lượng lớn; KHÔNG tự tải trong `Run all`.** Nếu thiếu, có thể sử dụng ô này sau khi kiểm tra dung lượng Drive. Nguồn 2014 5-core là một **tập con**, không bảo đảm bao phủ toàn bộ train interactions; notebook sẽ báo tỷ lệ match thật và không giả định reviews luôn tồn tại.

In [ ]:

DOWNLOAD_REVIEWS = False  # chỉ bật True nếu muốn tải file 2014 bản đầy đủ thật
if DOWNLOAD_REVIEWS:
    EXTERNAL.mkdir(parents=True,exist_ok=True)
    url='https://mcauleylab.ucsd.edu/public_datasets/data/amazon/categoryFiles/reviews_Books.json.gz'
    # -c tiếp tục download nếu file đã tồn tại; check gzip đầy đủ ở cell tiếp theo
    subprocess.run(['wget','-c','--show-progress','-O',str(RAW_REVIEWS),url],check=True)
print('Reviews:',RAW_REVIEWS, 'exists:',RAW_REVIEWS.is_file())


### 4. Index raw Amazon và chỉ nhận reviews của cặp tương tác train

**Bắt buộc:** `reviewerID` + `asin` phải khớp mapper và `(uid,iid)` phải có trong train. Nếu source có nhiều review cho cùng một cặp, bỏ review của cặp đó để tránh nhầm sự kiện. Dữ liệu review từ cùng cặp train **không chứng minh** event-level timestamp không rò rỉ; giới hạn này sẽ ghi vào manifest.

In [ ]:

assert RAW_METADATA.is_file(), f'Thiếu raw metadata: {RAW_METADATA}'
if REQUIRE_REVIEWS: assert RAW_REVIEWS.is_file(), f'Thiếu raw reviews: {RAW_REVIEWS}'
assert RAW_METADATA.stat().st_size>0
if RAW_REVIEWS.is_file():
    with gzip.open(RAW_REVIEWS,'rb') as f: assert f.read(1), 'File reviews rỗng/hỏng'

INDEX_MANIFEST=WORK/'raw_index/index_manifest.json'
BOOKS_FILE=WORK/'raw_index/books.jsonl'
REVIEWS_FILE=WORK/'raw_index/train_reviews.jsonl'
if INDEX_MANIFEST.is_file():
    man=json.loads(INDEX_MANIFEST.read_text())
    assert man['split_and_mapper_sha256']==split_hashes
    assert man['metadata_sha256']==sha256(RAW_METADATA)
    assert man['reviews_sha256']==(sha256(RAW_REVIEWS) if RAW_REVIEWS.is_file() else None)
    assert BOOKS_FILE.is_file() and REVIEWS_FILE.is_file()
    print('Reuse immutable raw index:',man['metadata_found'], '/', N_ITEMS)
else:
    src={}; stats=Counter()
    for row in text_rows(RAW_METADATA):
        asin=str(row.get('asin',''))
        iid=ASIN_TO_IID.get(asin)
        if iid is None:continue
        if iid in src:stats['duplicate_ASIN_metadata']+=1;continue
        description=clean_text(row.get('description'),MAX_DESCRIPTION_CHARS)
        features=clean_text(row.get('feature'),500)
        categories=clean_text(row.get('categories'),500)
        src[iid]={'iid':iid,'asin':asin,'title':clean_text(row.get('title'),500),
                  'description':description,'features':features,'categories':categories,
                  'metadata_found':True}
    found=len(src)
    assert found/N_ITEMS >= .95, f'Metadata coverage too low: {found/N_ITEMS:.2%}'
    for iid in range(N_ITEMS):
        if iid not in src:
            src[iid]={'iid':iid,'asin':ITEM_IDS[iid],'title':'None','description':'None',
                      'features':'None','categories':'None','metadata_found':False}

    # CSR membership dùng O(1) tra cứu, không đọc validation/test vào profile.
    train_pairs=set(zip(*TRN.nonzero()))
    train_reviews={};dup=set()
    if RAW_REVIEWS.is_file():
        for row in text_rows(RAW_REVIEWS):
            u=REVIEWER_TO_UID.get(str(row.get('reviewerID','')))
            i=ASIN_TO_IID.get(str(row.get('asin','')))
            if u is None or i is None:continue
            pair=(u,i)
            if pair not in train_pairs:
                stats['review_nontrain_pairs_rejected']+=1;continue
            if pair in dup:continue
            if pair in train_reviews:
                del train_reviews[pair];dup.add(pair);stats['ambiguous_review_pairs']+=1;continue
            train_reviews[pair]={'uid':u,'iid':i,'review':clean_text(row.get('reviewText'),MAX_REVIEW_CHARS),
                                'review_sha256':digest(json.dumps([u,i,row.get('reviewText',''),row.get('unixReviewTime')],default=str))}
    with BOOKS_FILE.open('w',encoding='utf-8') as f:
        for i in range(N_ITEMS):f.write(json.dumps(src[i],ensure_ascii=False)+'\n')
    with REVIEWS_FILE.open('w',encoding='utf-8') as f:
        for (u,i),rec in sorted(train_reviews.items()):f.write(json.dumps(rec,ensure_ascii=False)+'\n')
    man={'status':'PASS','source_type':'Amazon Books 2014 raw text','metadata_sha256':sha256(RAW_METADATA),
         'reviews_sha256':sha256(RAW_REVIEWS) if RAW_REVIEWS.is_file() else None,
         'split_and_mapper_sha256':split_hashes,'metadata_found':found,
         'missing_title':sum(x['title']=='None' for x in src.values()),
         'missing_description':sum(x['description']=='None' for x in src.values()),
         'train_pairs':len(train_pairs),'unambiguous_train_reviews':len(train_reviews),
         'train_review_coverage':len(train_reviews)/len(train_pairs),
         'ambiguous_review_pairs':len(dup),
         'nontrain_review_rows_rejected':stats['review_nontrain_pairs_rejected'],
         'review_source_policy':'TRAIN_PAIR_ONLY' if RAW_REVIEWS.is_file() else 'NO_REVIEWS_PILOT_ONLY',
         'limitations':['Review event timestamps are not reconstructible from binary matrices',
                        'Raw reviews must be verified against split, mapped train pairs may contain duplicate events']}
    atomic_json(INDEX_MANIFEST,man)
    print('INDEX CREATED')
print(json.dumps({k:man[k] for k in ['metadata_found','missing_title','missing_description','unambiguous_train_reviews','train_review_coverage','review_source_policy']},indent=2))
if REQUIRE_REVIEWS and man['unambiguous_train_reviews']==0:
    raise RuntimeError('Có file raw reviews nhưng không match train pair nào. Kiểm tra vintage/mapper trước khi tạo prompts.')


### 5. Tạo **một file JSONL** gồm 9.332 item prompts theo Amazon instruction của paper

Item prompt sử dụng `title`, `description`. **Nếu description thiếu:** chỉ tái dựng nội dung từ metadata features/categories và tối đa 3 review có bằng chứng thuộc train. Đây là *reconstruction*, không khẳng định là script batch nguyên bản của tác giả.

In [ ]:

ITEM_SYSTEM=(PAPER_REPO/'generation/instruction/amazon_item.txt').read_text(encoding='utf-8')
USER_SYSTEM=(PAPER_REPO/'generation/instruction/amazon_user.txt').read_text(encoding='utf-8')
assert 'summarization' in ITEM_SYSTEM and 'summarization' in USER_SYSTEM

books={r['iid']:r for r in read_jsonl(BOOKS_FILE)}
train_review_by_pair={(r['uid'],r['iid']):r for r in read_jsonl(REVIEWS_FILE)}
item_reviews=defaultdict(list)
for (u,i),r in sorted(train_review_by_pair.items()):
    if r['review']!='None':item_reviews[i].append(r)

def item_payload(iid):
    b=books[iid]
    desc=b['description'];source='raw_description'
    used=[]
    if desc=='None':
        facts=[]
        if b['features']!='None':facts.append('Features: '+b['features'])
        if b['categories']!='None':facts.append('Categories: '+b['categories'])
        selected=item_reviews[iid][:MISSING_DESC_REVIEW_COUNT]
        if selected:
            facts.extend('Verified train review: '+r['review'][:300] for r in selected)
            used=[[int(r['uid']),iid] for r in selected]
        desc=' '.join(facts)[:MAX_DESCRIPTION_CHARS] if facts else 'None'
        source='train_only_fallback' if facts else 'no_reliable_description'
    return {'title':b['title'],'description':desc},source,used

ITEM_PROMPTS=WORK/'prompts/item_prompts.jsonl'
rows=[]
for i in range(N_ITEMS):
    payload,src,used=item_payload(i)
    user_content='BOOK INFORMATION:\n'+json.dumps(payload,ensure_ascii=False)
    h=digest(ITEM_SYSTEM+'\nUSER\n'+user_content)
    rows.append({'kind':'item','id':i,'asin':books[i]['asin'],'system':ITEM_SYSTEM,
                 'user':user_content,'prompt_sha256':h,
                 'description_policy':src,'train_review_pairs_used':used,
                 'input_fields':['title','description']})
fingerprint=digest(''.join(r['prompt_sha256'] for r in rows))
payload={'kind':'item','count':N_ITEMS,'fingerprint':fingerprint,
         'system_sha256':digest(ITEM_SYSTEM),'raw_index_sha256':sha256(INDEX_MANIFEST),
         'history_cap':MAX_USER_HISTORY,'missing_desc_review_count':MISSING_DESC_REVIEW_COUNT}
require_or_freeze(WORK/'prompts/item_manifest.json',payload)
if ITEM_PROMPTS.is_file():
    old=list(read_jsonl(ITEM_PROMPTS));assert len(old)==N_ITEMS and all(a['prompt_sha256']==b['prompt_sha256'] for a,b in zip(old,rows)), 'Prompt changed, use new RUN_ID'
else:
    with ITEM_PROMPTS.open('w',encoding='utf-8') as f:
        for row in rows:f.write(json.dumps(row,ensure_ascii=False)+'\n')
print('ITEM PROMPTS FROZEN:',N_ITEMS,'Missing-description fallback:',Counter(r['description_policy'] for r in rows))
print('Ví dụ:',rows[0]['user'][:550])


### 6. Gemini qua DFLO — tạo **ITEM profiles** (pilot mặc định)

Mỗi lần chạy chỉ gọi tối đa `NEW_REQUEST_LIMIT` requests mới. Đã có kết quả hợp lệ sẽ **skip**; ghi token thực trả về. Khi kiểm tra 3 mẫu đầu tiên đạt yêu cầu, tăng `NEW_REQUEST_LIMIT` tùy quota. Tối đa profile phải có nội dung đáng tin cậy; `summarization == "None"` được đánh dấu `needs_review`, không âm thầm dùng làm embedding.

In [ ]:

from IPython.display import display
def secure_key(name):
    key=os.environ.get(name)
    if key:return key
    try:
        from google.colab import userdata
        key=userdata.get(name)
    except Exception:key=None
    return key or getpass(f'{name} (hidden): ')
def parse_gemini_response(raw):
    s=(raw or '').strip()
    s=re.sub(r'^```(?:json)?\s*','',s,flags=re.I)
    s=re.sub(r'\s*```$','',s)
    val=json.loads(s)
    assert isinstance(val,dict) and isinstance(val.get('summarization'),str) and isinstance(val.get('reasoning'),str)
    return {'summarization':val['summarization'].strip(),'reasoning':val['reasoning'].strip()}

def profile_file(kind):return WORK/'profiles'/f'{kind}.jsonl'
def existing_profiles(kind):
    out={}
    for rec in read_jsonl(profile_file(kind)) if profile_file(kind).is_file() else []:
        k=int(rec['id'])
        assert k not in out, f'Duplicate {kind} id {k} (do not use silently)'
        out[k]=rec
    return out

def generate_profiles(kind,new_limit=3):
    assert kind in ('item','user')
    prompt_path=WORK/'prompts'/f'{kind}_prompts.jsonl'
    assert prompt_path.is_file(),f'Chưa tạo prompts {kind}'
    prompts=list(read_jsonl(prompt_path));done=existing_profiles(kind)
    if not ALLOW_DFLO_CALLS:
        print(f'DRY MODE: {kind} ready {len(prompts)}, done {len(done)}; không gọi API.')
        for r in prompts[:2]:print('sample',r['id'],r['user'][:400])
        return
    assert new_limit is not None and 1 <= new_limit <= 500, 'Limit 1..500 mỗi lượt để kiểm soát chi phí'
    api_key=secure_key('DFLO_API_KEY')
    cli=OpenAI(api_key=api_key,base_url=DFLO_BASE_URL,max_retries=0,timeout=120)
    count=0
    for p in prompts:
        if p['id'] in done:
            assert done[p['id']]['prompt_sha256']==p['prompt_sha256'], f'Changed prompt {kind} {p["id"]}'
            assert done[p['id']]['requested_model']==DFLO_MODEL, f'Model changed in {kind} {p["id"]}'
            continue
        if count>=new_limit:break
        last=None
        for attempt in range(3):
            try:
                resp=cli.chat.completions.create(
                    model=DFLO_MODEL,
                    messages=[{'role':'system','content':p['system']}, {'role':'user','content':p['user']}],
                    temperature=0,max_tokens=900 if kind=='item' else 1400)
                parsed=parse_gemini_response(resp.choices[0].message.content)
                max_words=200 if kind=='item' else 100
                assert len(parsed['summarization'].split())<=max_words, 'summary too long'
                if kind == 'item':
                    assert len(parsed['reasoning'].split()) <= 200, 'item reasoning too long'
                last=None;break
            except Exception as exc:
                last=exc
                print(f'{kind} {p["id"]} attempt {attempt+1} error:',type(exc).__name__,str(exc)[:180])
                if attempt<2:time.sleep(2**attempt*5)
        if last is not None:raise RuntimeError(f'Không hoàn thành {kind} {p["id"]}. Đã checkpoint các mẫu trước.') from last
        u=resp.usage
        rec={'kind':kind,'id':p['id'],'prompt_sha256':p['prompt_sha256'],
             'requested_model':DFLO_MODEL,'returned_model':resp.model,
             'summarization':parsed['summarization'],'reasoning':parsed['reasoning'],
             'status':'ok' if parsed['summarization'] and parsed['summarization']!='None' else 'needs_review',
             'usage':{'input_tokens':getattr(u,'prompt_tokens',None),'output_tokens':getattr(u,'completion_tokens',None)},
             'created_utc':datetime.now(timezone.utc).isoformat()}
        with profile_file(kind).open('a',encoding='utf-8') as f:
            f.write(json.dumps(rec,ensure_ascii=False)+'\n');f.flush()
            try:os.fsync(f.fileno())
            except OSError:pass
        count+=1
        if MIN_SECONDS_BETWEEN_DFLO_CALLS>0:time.sleep(MIN_SECONDS_BETWEEN_DFLO_CALLS)
        print(kind,p['id'],rec['status'],'tokens',rec['usage'], 'model',resp.model)
    all_done=existing_profiles(kind)
    tokens_in=sum(x['usage']['input_tokens'] or 0 for x in all_done.values())
    tokens_out=sum(x['usage']['output_tokens'] or 0 for x in all_done.values())
    print('PROGRESS:',len(all_done),'/',len(prompts),'usage tokens IN/OUT',tokens_in,tokens_out)
    if DFLO_INPUT_USD_PER_M is not None and DFLO_OUTPUT_USD_PER_M is not None:
        print('DFLO estimate USD:',round((tokens_in*DFLO_INPUT_USD_PER_M+tokens_out*DFLO_OUTPUT_USD_PER_M)/1e6,4))
    else:print('DFLO actual bill: UNKNOWN (configure verified gateway rates).')

NEW_REQUEST_LIMIT=3
generate_profiles('item',new_limit=NEW_REQUEST_LIMIT)


### 7. Tạo user prompts SAU KHI có đủ 9.332 item profiles

Dùng **train histories** + **item profile Gemini mới** + review hợp lệ của chính user nếu tìm thấy. Không dùng item text profiles gốc của paper. **Mặc định dùng toàn bộ train items của user**. Chỉ khi tự đặt `MAX_USER_HISTORY=20` mới thực hiện lấy mẫu ổn định (reconstruction, phải ghi rõ trong báo cáo).

In [ ]:

item_profiles=existing_profiles('item')
assert len(item_profiles)==N_ITEMS, f'Cần đủ {N_ITEMS} item profiles, hiện {len(item_profiles)}'
item_prompts={r['id']:r for r in read_jsonl(ITEM_PROMPTS)}
for i in range(N_ITEMS):
    assert item_profiles[i]['status']=='ok',f'Item {i} insufficient source/profile; resolve first'
    assert item_profiles[i]['prompt_sha256']==item_prompts[i]['prompt_sha256']

USER_PROMPTS=WORK/'prompts/user_prompts.jsonl'
user_rows=[]
for uid in range(N_USERS):
    history=TRN.indices[TRN.indptr[uid]:TRN.indptr[uid+1]].tolist()
    if MAX_USER_HISTORY is not None and len(history)>MAX_USER_HISTORY:
        history=sorted(history,key=lambda iid:digest(f'user:{uid}:item:{iid}:seed:2026'))[:MAX_USER_HISTORY]
    entries=[];rv_pairs=[]
    for iid in history:
        item=books[int(iid)]
        review=train_review_by_pair.get((uid,int(iid)))
        entries.append({'title':item['title'],
                        'description':item_profiles[int(iid)]['summarization'],
                        'review':review['review'] if review else 'None'})
        if review:rv_pairs.append([uid,int(iid)])
    user_content='PURCHASED ITEMS:\n'+json.dumps(entries,ensure_ascii=False)
    h=digest(USER_SYSTEM+'\nUSER\n'+user_content)
    user_rows.append({'kind':'user','id':uid,'system':USER_SYSTEM,'user':user_content,
                      'prompt_sha256':h,'train_history_items':[int(i) for i in history],
                      'train_review_pairs_used':rv_pairs,
                      'note':'user history restricted to train; selected item summaries are from current Gemini run'})
hash_=digest(''.join(x['prompt_sha256'] for x in user_rows))
require_or_freeze(WORK/'prompts/user_manifest.json',
                  {'kind':'user','count':N_USERS,'fingerprint':hash_,
                   'system_sha256':digest(USER_SYSTEM),
                   'source_item_profile_file_sha256':sha256(profile_file('item')),
                   'history_cap':MAX_USER_HISTORY,
                   'raw_index_sha256':sha256(INDEX_MANIFEST)})
if USER_PROMPTS.is_file():
    old=list(read_jsonl(USER_PROMPTS))
    assert len(old)==N_USERS and all(a['prompt_sha256']==b['prompt_sha256'] for a,b in zip(old,user_rows)), 'User prompt changed; change RUN_ID'
else:
    with USER_PROMPTS.open('w',encoding='utf-8') as f:
        for x in user_rows:f.write(json.dumps(x,ensure_ascii=False)+'\n')
char_lengths=np.array([len(x['user']) for x in user_rows])
print('USER PROMPTS FROZEN:',len(user_rows),'chars p50/p95/max:',np.percentile(char_lengths,[50,95,100]).astype(int),'sample:',user_rows[0]['user'][:400])
print('Lưu ý: nếu user prompt dài quá quota, chỉnh MAX_USER_HISTORY và đổi RUN_ID trước khi gọi API.')


### 8. Gemini qua DFLO — tạo **USER profiles**

Nhớ bật `ALLOW_DFLO_CALLS=True` trong ô cấu hình khi thật sự cho phép gọi API. Chạy pilot 3 người dùng trước, sau đó tăng `NEW_REQUEST_LIMIT` khi đã kiểm tra.

In [ ]:

NEW_REQUEST_LIMIT=3
generate_profiles('user',new_limit=NEW_REQUEST_LIMIT)


### 9. OpenAI `text-embedding-ada-002` — chỉ embed trường `summarization`

Tự lưu **các shard 32 profiles** để resume. `ALLOW_OPENAI_CALLS=False` mặc định. Không dùng DFLO key; cần OpenAI Platform key. Mã hóa profiles thành 1536 chiều theo mô hình paper; ghi token usage từ API.

In [ ]:

def check_complete_profiles():
    out={}
    for kind,n in [('item',N_ITEMS),('user',N_USERS)]:
        ps=existing_profiles(kind)
        assert len(ps)==n,f'{kind} mới có {len(ps)}/{n} profiles'
        assert set(ps)==set(range(n)),f'{kind}: id not contiguous 0..{n-1}'
        bad=[i for i,x in ps.items() if x['status']!='ok' or not x['summarization']]
        assert not bad,f'{kind} invalid profiles, first: {bad[:10]}'
        out[kind]=ps
    return out

def embed_kind(kind,batches_limit=1,chunk_size=32):
    all_profiles=check_complete_profiles()
    assert kind in ('item','user') and 1<=batches_limit<=50
    ps=all_profiles[kind];n=len(ps)
    outdir=WORK/'embedding_parts'/kind;outdir.mkdir(parents=True,exist_ok=True)
    prompt_profile_fingerprint=digest(''.join(ps[i]['prompt_sha256']+ps[i]['summarization'] for i in range(n)))
    require_or_freeze(outdir/'embedding_manifest.json',
        {'model':OPENAI_EMBED_MODEL,'n':n,'dimension':1536,'chunk_size':chunk_size,
         'source_profiles_sha256':sha256(profile_file(kind)),
         'source_content_fingerprint':prompt_profile_fingerprint})
    pending=[]
    for start in range(0,n,chunk_size):
        p=outdir/f'{start:05d}.npz'
        if p.exists():
            with np.load(p,allow_pickle=False) as z:
                assert np.array_equal(z['ids'],np.arange(start,min(n,start+chunk_size)))
                assert z['embeddings'].shape==(min(chunk_size,n-start),1536)
            continue
        pending.append(start)
    print(kind,'done parts', (n+chunk_size-1)//chunk_size-len(pending),'pending',len(pending))
    if not ALLOW_OPENAI_CALLS:
        print('DRY MODE: không gọi OpenAI. Next chunks:',pending[:3]);return
    key=secure_key('OPENAI_API_KEY')
    api=OpenAI(api_key=key,max_retries=2,timeout=120)
    for start in pending[:batches_limit]:
        ids=list(range(start,min(n,start+chunk_size)))
        txt=[ps[i]['summarization'] for i in ids]
        res=api.embeddings.create(model=OPENAI_EMBED_MODEL,input=txt)
        assert len(res.data)==len(ids)
        vec=np.array([x.embedding for x in sorted(res.data,key=lambda x:x.index)],dtype=np.float32)
        assert vec.shape==(len(ids),1536) and np.isfinite(vec).all()
        assert np.all(np.linalg.norm(vec,axis=1)>0)
        path=outdir/f'{start:05d}.npz'
        assert not path.exists(),f'Already exists {path}'
        temp=outdir/f'{start:05d}.temp.npz'
        np.savez_compressed(temp,ids=np.array(ids),embeddings=vec)
        temp.replace(path)
        tokens=getattr(res.usage,'total_tokens',None)
        with (WORK/'logs'/f'openai_{kind}.jsonl').open('a',encoding='utf-8') as f:
            f.write(json.dumps({'model':res.model,'start':start,'count':len(ids),
                                'total_tokens':tokens,'estimated_usd':None if tokens is None else tokens*ADA002_USD_PER_M/1e6})+'\n')
        print('Saved',path.name,'rows',len(ids),'tokens',tokens,'estimated USD',None if tokens is None else round(tokens*ADA002_USD_PER_M/1e6,7))

EMBED_BATCHES_PER_RUN=1  # chạy thử 32 profiles đầu, tăng tối đa 50 batch/lượt
embed_kind('item',batches_limit=EMBED_BATCHES_PER_RUN)
# Khi item đã tạo hết, chạy thêm:
# embed_kind('user',batches_limit=EMBED_BATCHES_PER_RUN)


### 10. Đóng gói thành 7 `.pkl` và QA bắt buộc

Chỉ thực thi khi **đã có đủ profiles + đủ embeddings**. Giữ nguyên 3 ma trận gốc theo SHA256. Tạo `usr_prf.pkl`, `itm_prf.pkl` dạng list dict `{profile, reasoning}` đúng `data/read_profile.py` của tác giả; `usr_emb_np.pkl`, `itm_emb_np.pkl` dạng `numpy.float32` `(11000,1536)`, `(9332,1536)`.

Output: `Data_SNA_Group14/experiments_gemini/<RUN_ID>/dataset/amazon/`. Không đụng vào `Data_SNA_Group14/data/raw/amazon/`.

In [ ]:

def assemble_embeddings(kind,n,chunk_size=32):
    folder=WORK/'embedding_parts'/kind
    man=json.loads((folder/'embedding_manifest.json').read_text())
    assert man['model']==OPENAI_EMBED_MODEL and man['n']==n and man['chunk_size']==chunk_size
    result=np.empty((n,1536),dtype=np.float32)
    for start in range(0,n,chunk_size):
        path=folder/f'{start:05d}.npz'
        assert path.is_file(),f'Missing embedding shard {kind}:{start}'
        with np.load(path,allow_pickle=False) as z:
            ids=z['ids'];v=z['embeddings']
            assert np.array_equal(ids,np.arange(start,min(n,start+chunk_size)))
            assert v.shape==(len(ids),1536)
            result[ids]=v
    assert np.isfinite(result).all() and np.all(np.linalg.norm(result,axis=1)>0)
    return result

def package_dataset():
    profiles=check_complete_profiles()
    arrays={'itm_emb_np.pkl':assemble_embeddings('item',N_ITEMS),
            'usr_emb_np.pkl':assemble_embeddings('user',N_USERS)}
    arrays.update({'itm_prf.pkl':[{'profile':profiles['item'][i]['summarization'],
                                  'reasoning':profiles['item'][i]['reasoning']} for i in range(N_ITEMS)],
                   'usr_prf.pkl':[{'profile':profiles['user'][i]['summarization'],
                                  'reasoning':profiles['user'][i]['reasoning']} for i in range(N_USERS)]})
    dest=WORK/'dataset/amazon';dest.mkdir(parents=True,exist_ok=True)
    man=json.loads(INDEX_MANIFEST.read_text(encoding='utf-8'))
    assert not (dest/'dataset_manifest.json').exists(), 'Dataset already frozen: không ghi đè. Dùng RUN_ID mới.'
    for name in ['trn_mat.pkl','val_mat.pkl','tst_mat.pkl']:
        from_path=RAW/name;to_path=dest/name
        if not to_path.exists():shutil.copy2(from_path,to_path)
        assert sha256(from_path)==sha256(to_path)==split_hashes[name]
    for name,obj in arrays.items():
        target=dest/name
        assert not target.exists(),f'{target} exists; no overwrite'
        tmp=dest/(name+'.tmp')
        with tmp.open('wb') as f:pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
        tmp.replace(target)
    for name in arrays:
        with (dest/name).open('rb') as f:obj=pickle.load(f)
        if name.endswith('emb_np.pkl'):
            assert obj.shape==arrays[name].shape and obj.dtype==np.float32
        else:assert len(obj)==len(arrays[name]) and all('profile' in p and 'reasoning' in p for p in obj)
    report={'status':'PASS','run_id':RUN_ID,'source':'Gemini (DFLO) profile summaries + OpenAI ada-002 embeddings',
            'profile_model_request':DFLO_MODEL,'embedding_model':OPENAI_EMBED_MODEL,
            'split_hashes_original':split_hashes,
            'output_sha256':{name:sha256(dest/name) for name in ['trn_mat.pkl','val_mat.pkl','tst_mat.pkl',*arrays]},
            'shapes':{'user':[N_USERS,1536],'item':[N_ITEMS,1536]},
            'review_source_policy':man['review_source_policy'],
            'note':'Input profile reconstruction; event-level provenance cannot be proven from matrix pairs alone.'}
    atomic_json(dest/'dataset_manifest.json',report)
    print('DATASET PACKAGE PASS:',dest,'\nNext: 01_PREPARE_GROUP_EXPERIMENTS_Colab.ipynb')
    return report

DO_PACKAGE=False
if DO_PACKAGE: package_dataset()
else:print('Packaging disabled by default. Set DO_PACKAGE=True only after all profiles/embeddings are complete.')


### 11. Tổng hợp tiến độ và lượng tokens (chạy bất cứ lúc nào)

In [ ]:

for kind,n in [('item',N_ITEMS),('user',N_USERS)]:
    done=existing_profiles(kind)
    print(kind, 'profiles',len(done),'/',n,
          '| bad status',sum(r['status']!='ok' for r in done.values()))
    t_in=sum((x['usage'].get('input_tokens') or 0) for x in done.values())
    t_out=sum((x['usage'].get('output_tokens') or 0) for x in done.values())
    print('  DFLO tokens',t_in,'input',t_out,'output')
    folder=WORK/'embedding_parts'/kind
    print('  embedding shards:',len(list(folder.glob('*.npz'))) if folder.exists() else 0)
print('Dataset packaged:',(WORK/'dataset/amazon/dataset_manifest.json').exists())


---
### Giải thích hạn chế (đưa vào báo cáo)

- Mã nguồn công khai RLMRec không kèm toàn bộ Amazon batch prompt reconstruction. Vì vậy cần báo rõ chính sách tùy chọn giới hạn số train items và fallback cho sách thiếu description.
- User reviews chỉ dùng nếu map được vào cặp train, và không khẳng định có event-level timestamp purity vì ma trận nhị phân không chứa review event IDs.
- `DFLO_MODEL` là alias gateway, phải lưu `returned_model` thật ở từng response.
- Không nên làm thay đổi prompt/profile đã phát sinh chi phí trong cùng RUN_ID. Khi sửa policy, tạo RUN_ID mới.
- Sau khi đóng gói, dùng notebook thứ hai tạo bản **được điều chỉnh đường dẫn** của notebooks nhóm; không viết lại E1–E4.